# M3: Graph and KG-RAG (KGRAG)
**Your job:** take M2's facts (`triples.jsonl`), build the knowledge graph, and test the search that walks the graph.
Every section below says what it does and what you should see. Run the cells from top to bottom, one at a time.

## 1. Upload the project ZIP and fix the folders

In [ ]:
from google.colab import files
import zipfile, os, shutil

uploaded = files.upload()          # choose kgrag_m2_colab_latest.zip (the full project)
zip_name = next(n for n in uploaded if n.endswith(".zip"))
with zipfile.ZipFile(zip_name) as z:
    z.extractall("/content/kgrag")

# the zip was made on Windows: move files with backslashes in their names into real folders
root = "/content/kgrag"
for name in os.listdir(root):
    if "\\" in name:
        new_path = os.path.join(root, name.replace("\\", "/"))
        os.makedirs(os.path.dirname(new_path), exist_ok=True)
        shutil.move(os.path.join(root, name), new_path)
%cd /content/kgrag
!ls team/M3_graph

## 1b. Upload the updated search code (`retriever.py`)
This is the one file that was changed: it adds an OPTIONAL Sentence-BERT fallback for linking names. It is off by default, so nobody else's results change.
Choose `retriever.py` when asked.

In [ ]:
from google.colab import files
import shutil
up = files.upload()                                   # choose retriever.py
shutil.copy(next(n for n in up if n.endswith(".py")), "/content/kgrag/src/kgrag/graph/retriever.py")
!grep -n "link_fallback" /content/kgrag/src/kgrag/graph/retriever.py | head -3

## 2. Connect Google Drive (results are saved there so a disconnect cannot delete them)

In [ ]:
from google.colab import drive
import os
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")
OUT_DIR = "/content/drive/MyDrive/m3_outputs"
os.makedirs(OUT_DIR, exist_ok=True)
print("Output folder:", OUT_DIR)

## 3. Put M2's facts file in place
M3 reads `team/M2_extraction/outputs/triples.jsonl`. Either upload the file M2 made, or copy it from Drive.
**Expected: the line count printed below is 14134.** If it is 24, 244 or 254, you have the wrong file.

In [ ]:
import os, shutil
TARGET = "/content/kgrag/team/M2_extraction/outputs/triples.jsonl"
os.makedirs(os.path.dirname(TARGET), exist_ok=True)

FROM_DRIVE = "/content/drive/MyDrive/m2_outputs/triples.jsonl"      # where you saved it after the M2 run
if os.path.exists(FROM_DRIVE):
    shutil.copy(FROM_DRIVE, TARGET)
    print("copied from Drive")
else:
    from google.colab import files
    up = files.upload()                                              # choose triples.jsonl
    shutil.copy(next(iter(up)), TARGET)
!wc -l /content/kgrag/team/M2_extraction/outputs/triples.jsonl
!head -n 3 /content/kgrag/team/M2_extraction/outputs/triples.jsonl

## 4. Install the tools (a few minutes)

In [ ]:
!pip install -q sentence-transformers spacy networkx numpy
!python -m spacy download en_core_web_sm -q
import torch
print("GPU available:", torch.cuda.is_available())

## 5. Sanity check on the toy data (no models)
Expect: `OK` under contract C3 at the end.

In [ ]:
%cd /content/kgrag
!python team/M3_graph/run_m3.py 2>&1 | tail -20

## 6. Settings used by every cell below
Change `EMB` or `NER` here only. `sbert` and `spacy` are the real models.

In [ ]:
import sys, os, csv, json, time, collections
ROOT = "/content/kgrag"
sys.path.insert(0, ROOT + "/src")
os.chdir(ROOT)

EMB, NER = "sbert", "spacy"
DOCS = "data/hotpot/docs.json"
QUESTIONS = "data/hotpot/questions.json"
TRIPLES = "team/M2_extraction/outputs/triples.jsonl"
HOPS, BUDGET = 2, 120

from kgrag.config import Config
from kgrag.data import load_docs, load_questions
from kgrag.evaluation.metrics import supporting_recall
from kgrag.extraction.entities import entity_key
from kgrag.extraction.pipeline import load_triples
from kgrag.factory import make_embedder, make_ner
from kgrag.graph.kg import KnowledgeGraph
from kgrag.graph.retriever import GraphRetriever

cfg = Config(ner=NER, embedder=EMB, hops=HOPS, token_budget=BUDGET)
docs, qs = load_docs(DOCS), load_questions(QUESTIONS)
triples = load_triples(TRIPLES)
print(len(docs), "documents |", len(qs), "questions |", len(triples), "triples")

## 7. The graph: size, hubs, connectivity (numbers for Table 1 of the paper)
- **nodes** = distinct entities, **edges** = facts.
- **Hubs** are nodes with huge numbers of connections. They join unrelated things and add noise.
- **Components** are separate islands. If the graph is mostly tiny islands, walking it cannot join two pages.

In [ ]:
import networkx as nx
kg = KnowledgeGraph(triples)
G = kg.G
print(kg.stats(), "| average connections per node: %.2f" % (2 * G.number_of_edges() / max(G.number_of_nodes(), 1)))

print("\nTop 15 hubs (name, connections):")
for n, d in sorted(G.degree, key=lambda x: -x[1])[:15]:
    print(f"  {d:5d}  {n}")

comps = sorted((len(c) for c in nx.weakly_connected_components(G)), reverse=True)
print(f"\nconnected islands: {len(comps)} | biggest island: {comps[0]} nodes | islands with only 2 nodes: {sum(1 for c in comps if c == 2)}")

rels = collections.Counter(d["relation"] for _, _, d in G.edges(data=True))
print("\nrelations in the graph:", rels.most_common(15))

## 8. Nodes that might be the same entity (tell M2, and decide the merge rule)
Too many merges make false paths. Too few leave broken paths. Read the list and note the pattern for your report.

In [ ]:
import difflib
blocks = {}
for n in G.nodes:
    blocks.setdefault(entity_key(n)[:3], []).append(n)
pairs = []
for b in blocks.values():
    for i, x in enumerate(b):
        for y in b[i + 1:]:
            s = difflib.SequenceMatcher(None, x.lower(), y.lower())
            if s.real_quick_ratio() > 0.85 and s.quick_ratio() > 0.85 and s.ratio() > 0.85:
                pairs.append((x, y))
print(len(pairs), "possible duplicate pairs. First 40:")
for x, y in pairs[:40]:
    print("  ", x, "  <->  ", y)

## 9. Main test: does the graph search find the needed documents? (no language model needed)
This is the real run. It prints one line per question and the **mean supporting recall**, which is the share of the needed pages that the graph search found.
Also saves `kg_retrieval.csv` and the hop ablation (1, 2, 3 hops).

In [ ]:
!python team/M3_graph/run_m3.py --embedder {EMB} --ner {NER} --docs {DOCS} --questions {QUESTIONS} --hops {HOPS} --budget {BUDGET} 2>&1 | tee /content/drive/MyDrive/m3_outputs/m3_main_log.txt | tail -60

## 10. Hub ablation (an extra experiment from the M3 README)
We remove nodes that have more than N connections, then re-test. If recall goes up, hubs were adding noise.
`None` means no filtering (the baseline).

In [ ]:
ner = make_ner(cfg, [d.title for d in docs])
emb = make_embedder(cfg)
emb.fit([f"{t.subject} {t.relation} {t.object}" for t in triples])

deg = dict(KnowledgeGraph(triples).G.degree)

def evaluate(tr, hops=HOPS, budget=BUDGET, **kw):
    kg_ = KnowledgeGraph(tr)
    ner.add(kg_.nodes)
    gr_ = GraphRetriever(kg_, ner, emb, hops, cfg.hop_penalty, budget, **kw)
    rows, nolink = [], 0
    for q in qs:
        r = gr_.retrieve(q["question"])
        nolink += 0 if r.debug.get("linked") else 1
        rows.append((q["type"], supporting_recall(q["supporting_titles"], r.titles)))
    out = {"all": sum(x for _, x in rows) / len(rows), "nothing_linked": nolink}
    for t in ("bridge", "comparison"):
        v = [x for tt, x in rows if tt == t]
        out[t] = sum(v) / len(v) if v else float("nan")
    return out

print(f"{'max connections':>16} | {'edges kept':>10} | recall all | bridge | comparison")
hub_rows = []
for cap in (None, 300, 100, 30):
    tr = triples if cap is None else [t for t in triples if deg.get(t.subject, 0) <= cap and deg.get(t.object, 0) <= cap]
    t0 = time.time()
    r = evaluate(tr)
    hub_rows.append({"max_degree": cap, "edges": len(tr), **r})
    print(f"{str(cap):>16} | {len(tr):>10} | {r['all']:.3f}      | {r['bridge']:.3f}  | {r['comparison']:.3f}   ({time.time()-t0:.0f}s)")

## 11. Budget ablation
The budget is how many words of facts the language model gets. More words = more facts found, but more noise for the model.

In [ ]:
print(f"{'budget (words)':>14} | recall all | bridge | comparison")
bud_rows = []
for b in (60, 120, 200):
    r = evaluate(triples, budget=b)
    bud_rows.append({"budget": b, **r})
    print(f"{b:>14} | {r['all']:.3f}      | {r['bridge']:.3f}  | {r['comparison']:.3f}")

## 12. Normalisation ablation (names that are the same thing = one node)
Normalisation decides which spellings become ONE node. Too few merges break paths; too many create false paths.
M2 already merged names when it made `triples.jsonl`, so from this file we can only test **merging MORE**:
- **as M2 made it:** the baseline.
- **+ company words:** also treat "Google", "Google Inc" and "Google LLC" as one.
- **looser fuzzy 0.88:** also merge names that are 88% similar (default was 93%).
- **both.**
Read the sample merges printed under each row: if you see different things merged (for example two different dates), that variant creates false paths.
(Testing merging LESS needs M2 to re-run with normalisation off, about 50 minutes. Mention that as future work.)

In [ ]:
from collections import Counter
from kgrag.extraction.normalize import EntityNormalizer
from kgrag.extraction.types import Triple

def renormalise(tr, **kw):
    counts = Counter()
    for t in tr:
        counts[t.subject] += 1
        counts[t.object] += 1
    norm = EntityNormalizer(preferred=[d.title for d in docs], **kw)
    mp = norm.fit(counts)
    seen, out = set(), []
    for t in tr:
        nt = Triple(mp[t.subject], t.relation, mp[t.object], t.sentence, t.doc_title)
        k = (nt.subject, nt.relation, nt.object)
        if k not in seen:
            seen.add(k)
            out.append(nt)
    return out, norm.merges

variants = [("as M2 made it", None),
            ("+ company words", dict(merge_corporate_suffix=True)),
            ("looser fuzzy 0.88", dict(fuzzy_cutoff=0.88)),
            ("both", dict(merge_corporate_suffix=True, fuzzy_cutoff=0.88))]
norm_rows = []
print(f"{'variant':>20} | nodes | edges | merges | recall all | bridge | comparison")
for name, kw in variants:
    tr, merges = (triples, []) if kw is None else renormalise(triples, **kw)
    nodes = len({x for t in tr for x in (t.subject, t.object)})
    r = evaluate(tr)
    norm_rows.append({"variant": name, "nodes": nodes, "edges": len(tr), "merges": len(merges), **r})
    print(f"{name:>20} | {nodes:5d} | {len(tr):5d} | {len(merges):6d} | {r['all']:.3f}      | {r['bridge']:.3f}  | {r['comparison']:.3f}")
    for canon, forms in merges[:6]:
        print("      merged:", canon, "<-", forms[:3])

## 13. Sentence-BERT fallback for linking names (the "no entity linked" questions)
Normally a name in the question must match a node's name (almost) exactly. With the fallback ON, a name that matches nothing is linked to the node whose **name means the closest thing**.
- **threshold** = how close it must be. High (0.85) = careful, few links. Low (0.65) = many links, some wrong.
- **question-level** = if nothing at all got linked, use the nodes whose names are closest to the whole question.
`nothing_linked` should go DOWN. Then check whether recall goes UP. If it doesn't, the new links are wrong ones, which is also a result to report.

In [ ]:
fb_variants = [("fallback OFF", {}),
               ("mention, thr 0.85", dict(link_fallback=True, fallback_threshold=0.85)),
               ("mention, thr 0.75", dict(link_fallback=True, fallback_threshold=0.75)),
               ("mention, thr 0.65", dict(link_fallback=True, fallback_threshold=0.65)),
               ("mention 0.75 + question-level top2", dict(link_fallback=True, fallback_threshold=0.75,
                                                           question_fallback_topk=2, question_fallback_threshold=0.40))]
fb_rows = []
print(f"{'variant':>36} | nothing linked | recall all | bridge | comparison")
for name, kw in fb_variants:
    t0 = time.time()
    r = evaluate(triples, **kw)
    fb_rows.append({"variant": name, **r})
    print(f"{name:>36} | {r['nothing_linked']:14d} | {r['all']:.3f}      | {r['bridge']:.3f}  | {r['comparison']:.3f}   ({time.time()-t0:.0f}s)")

## 14. Explain every failure (the "you are done when" rule)
For each question where a needed page was **not** found, we label why:
1. **no entity linked**: the graph search could not match any name in the question to a node.
2. **a needed page has no facts**: M2 extracted nothing from that page, so the graph cannot reach it. This is M2's gap, not M3's.
3. **facts exist but were not retrieved**: the page is in the graph, but ranking, the hop limit or the budget missed it.

In [ ]:
kg = KnowledgeGraph(triples)
ner.add(kg.nodes)
gr = GraphRetriever(kg, ner, emb, HOPS, cfg.hop_penalty, BUDGET)
titles_with_facts = {t.doc_title for t in triples}

fail_rows, cats = [], collections.Counter()
for q in qs:
    r = gr.retrieve(q["question"])
    rec = supporting_recall(q["supporting_titles"], r.titles)
    if rec >= 1.0:
        cats["fully found"] += 1
        continue
    linked, unlinked = r.debug.get("linked", []), r.debug.get("unlinked", [])
    missing = [t for t in q["supporting_titles"] if t not in r.titles]
    if not linked:
        cat = "1. no entity linked"
    elif any(t not in titles_with_facts for t in missing):
        cat = "2. a needed page has no facts (M2 gap)"
    else:
        cat = "3. facts exist but were not retrieved"
    cats[cat] += 1
    fail_rows.append({"qid": q["id"], "type": q["type"], "recall": round(rec, 2), "question": q["question"],
                      "missing_pages": " | ".join(missing), "linked": " | ".join(linked),
                      "unlinked": " | ".join(unlinked), "category": cat, "MANUAL_NOTE": ""})

print("Question outcomes (put this in the paper):")
for k, v in sorted(cats.items()):
    print(f"  {v:4d}  {k}")
for row in fail_rows[:12]:
    print("\n-", row["question"], "\n   missing:", row["missing_pages"], "| linked:", row["linked"], "| ->", row["category"])

## 15. Save everything to Drive and download a copy

In [ ]:
import csv, shutil, os
def save(rows, name):
    if rows:
        with open(f"{OUT_DIR}/{name}", "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
save(hub_rows, "hub_ablation.csv")
save(bud_rows, "budget_ablation.csv")
save(norm_rows, "normalisation_ablation.csv")
save(fb_rows, "sbert_fallback_ablation.csv")
save(fail_rows, "failure_analysis.csv")
shutil.copy("team/M3_graph/outputs/kg_retrieval.csv", OUT_DIR + "/kg_retrieval.csv")
!ls -la /content/drive/MyDrive/m3_outputs
!zip -r m3_outputs.zip team/M3_graph /content/drive/MyDrive/m3_outputs -q
from google.colab import files
files.download("m3_outputs.zip")